# Automobile Dataset Analysis

This notebook explores the main factors associated with automobile fuel efficiency.

The analysis follows a simple progression: first understand the dataset, then explore relationships between variables, investigate the 24 MPG CAFE threshold, use PCA to summarize the mechanical structure of the cars, and finally test how well MPG can be predicted with linear regression.

The goal is not to build the most complex model possible, but to extract clear and interpretable information from the data.

## 1. Setup and data loading

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

sns.set_context("notebook")

In [3]:
# Load the automobile dataset
Automobiledf = pd.read_csv("automobiles.csv")

print(f"Dataset shape: {Automobiledf.shape[0]} rows × {Automobiledf.shape[1]} columns")
Automobiledf.head()

Dataset shape: 392 rows × 9 columns


,miles_per_gallon,cylinders,displacement,horsepower,weight_lbs,acceleration,model_year,origin_country,car_name
0,18.0,8,307.0,130,3504,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165,3693,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150,3436,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150,3433,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140,3449,10.5,70,1,ford torino


## 2. Dataset overview

The dataset contains 392 cars described by technical, performance and origin-related variables.  
I first check the variable types, missing values, and the distribution of cars across origins.

In [4]:
print("Column types:")
display(Automobiledf.dtypes.to_frame("dtype"))

print(f"\nTotal missing values: {Automobiledf.isna().sum().sum()}")

origin_labels = {1: "USA", 2: "Europe", 3: "Japan"}
origin_counts = (
    Automobiledf["origin_country"]
    .map(origin_labels)
    .value_counts()
    .reindex(["USA", "Europe", "Japan"])
)

print("\nCars by origin:")
display(origin_counts.to_frame("count"))

Column types:


,dtype
miles_per_gallon,float64
cylinders,int64
displacement,float64
horsepower,int64
weight_lbs,int64
acceleration,float64
model_year,int64
origin_country,int64
car_name,object



Total missing values: 0

Cars by origin:


,count
origin_country,
USA,245
Europe,68
Japan,79


`car_name` and `origin_country` are categorical variables, so the descriptive statistics below focus on the numerical variables.  
The dataset is clearly unbalanced by origin, with American cars representing most observations; this is why several analyses are also repeated by country.

In [5]:
num_cols = [
    "miles_per_gallon",
    "cylinders",
    "displacement",
    "horsepower",
    "weight_lbs",
    "acceleration",
    "model_year",
]

cat_cols = ["origin_country", "car_name"]

stats_global = (
    Automobiledf[num_cols]
    .describe()
    .loc[["mean", "std", "min", "max"]]
    .round(2)
)

stats_global

,miles_per_gallon,cylinders,displacement,horsepower,weight_lbs,acceleration,model_year
mean,23.45,5.47,194.41,104.47,2977.58,15.54,75.98
std,7.81,1.71,104.64,38.49,849.40,2.76,3.68
min,9.00,3.00,68.00,46.00,1613.00,8.00,70.00
max,46.60,8.00,455.00,230.00,5140.00,24.80,82.00


In [6]:
# Descriptive statistics by country of origin
stats_by_origin = {}

for code_value, country_name in origin_labels.items():
    stats_by_origin[country_name] = (
        Automobiledf.loc[Automobiledf["origin_country"] == code_value, num_cols]
        .describe()
        .loc[["mean", "std", "min", "max"]]
        .round(2)
    )

for country_name, country_stats in stats_by_origin.items():
    print(f"\n{country_name}")
    display(country_stats)


USA


,miles_per_gallon,cylinders,displacement,horsepower,weight_lbs,acceleration,model_year
mean,20.03,6.28,247.51,119.05,3372.49,14.99,75.59
std,6.44,1.66,98.38,39.90,795.35,2.74,3.66
min,9.00,4.00,85.00,52.00,1800.00,8.00,70.00
max,39.00,8.00,455.00,230.00,5140.00,22.20,82.00



Europe


,miles_per_gallon,cylinders,displacement,horsepower,weight_lbs,acceleration,model_year
mean,27.60,4.16,109.63,80.56,2433.47,16.79,75.68
std,6.58,0.51,22.69,20.16,491.81,3.09,3.42
min,16.20,4.00,68.00,46.00,1825.00,12.20,70.00
max,44.30,6.00,183.00,133.00,3820.00,24.80,82.00



Japan


,miles_per_gallon,cylinders,displacement,horsepower,weight_lbs,acceleration,model_year
mean,30.45,4.10,102.71,79.84,2221.23,16.17,77.44
std,6.09,0.59,23.14,17.82,320.50,1.95,3.65
min,18.00,3.00,70.00,52.00,1613.00,11.40,70.00
max,46.60,6.00,168.00,132.00,2930.00,21.00,82.00


A first pattern already appears: Japanese and European cars have higher average MPG, while American cars are heavier and tend to have larger engines.  
Before modeling anything, it is useful to see how these variables relate to each other.